# Implementation: Schrijver & Zwaan (2000) — Solar and Stellar Magnetic Activity
# 구현: Schrijver & Zwaan (2000) — 태양 및 항성 자기 활동

**English.** This notebook reproduces three of the monograph's central empirical anchors as numerical demonstrations: (1) Skumanich's spin-down law $\Omega \propto t^{-1/2}$, (2) a Babcock–Leighton-style butterfly diagram from a simple flux-transport toy model, (3) the activity–rotation diagram with saturation, (4) the universal F_X – Φ power-law relation, and (5) magnetic-carpet replacement-time statistics.

**한국어.** 이 노트북은 본 단행본의 세 가지 중심 경험적 정초를 수치적으로 재현합니다. (1) Skumanich 회전 감속 법칙 $\Omega \propto t^{-1/2}$, (2) 단순한 플럭스 수송 모형으로부터의 Babcock–Leighton 형 나비 도표, (3) 포화를 포함한 활동–회전 다이어그램, (4) 보편적 F_X – Φ 멱법칙, (5) 자기 카펫 갱신 시간 통계.

In [ ]:
"""Imports and global plotting style."""
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm

rng = np.random.default_rng(seed=20260427)
plt.rcParams["figure.figsize"] = (8.0, 4.5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

## Part 1 — Skumanich Spin-Down Law / Skumanich 회전 감속 법칙

**English.** Solving $dJ/dt = -K\Omega^3$ with $J = I\Omega$ yields $\Omega(t) = \Omega_0 (1 + 2K\Omega_0^2 t/I)^{-1/2}$. For long times this asymptotes to $\Omega \propto t^{-1/2}$ — Skumanich's law (Ch. 11).

**한국어.** $dJ/dt = -K\Omega^3$를 $J = I\Omega$로 풀면 $\Omega(t) = \Omega_0 (1 + 2K\Omega_0^2 t/I)^{-1/2}$이며, 장시간 점근적으로 $\Omega \propto t^{-1/2}$ — Skumanich 법칙(11장).

In [ ]:
def skumanich_omega(t_gyr: np.ndarray, omega0: float, tau_brake_gyr: float) -> np.ndarray:
    """Compute angular velocity vs. time under wind braking.

    Args:
        t_gyr: Time in Gyr.
        omega0: Initial angular velocity (rad s^-1).
        tau_brake_gyr: Braking timescale 2K*omega0^2/I (Gyr^-1) inverse.

    Returns:
        Omega(t) in rad s^-1.
    """
    return omega0 / np.sqrt(1.0 + t_gyr / tau_brake_gyr)

OMEGA_SUN = 2.87e-6  # rad s^-1, equatorial
T_SUN_GYR = 4.6
# Calibrate tau_brake so Omega(4.6 Gyr) = OMEGA_SUN given Omega(0.1 Gyr) ~ 5*OMEGA_SUN
OMEGA0 = 5.0 * OMEGA_SUN
TAU_BRAKE = (T_SUN_GYR - 0.1) / ((OMEGA0 / OMEGA_SUN) ** 2 - 1.0)

t_grid = np.linspace(0.05, 10.0, 400)
omega_grid = skumanich_omega(t_grid, OMEGA0, TAU_BRAKE)
p_rot_days = (2 * np.pi / omega_grid) / 86400.0

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
axes[0].loglog(t_grid, omega_grid / OMEGA_SUN, lw=2)
axes[0].axhline(1.0, color="k", ls=":", lw=1)
axes[0].axvline(T_SUN_GYR, color="k", ls=":", lw=1)
axes[0].set_xlabel("Age t [Gyr]")
axes[0].set_ylabel(r"$\Omega / \Omega_\odot$")
axes[0].set_title("Skumanich law (log-log)")

axes[1].plot(t_grid, p_rot_days, lw=2)
for label, age, prot in [("Pleiades", 0.13, 1.7), ("Hyades", 0.6, 5.0), ("Sun", 4.6, 25.0)]:
    axes[1].plot(age, prot, "o", ms=8)
    axes[1].annotate(label, (age, prot), textcoords="offset points", xytext=(8, 4))
axes[1].set_xlabel("Age [Gyr]")
axes[1].set_ylabel("P_rot [days]")
axes[1].set_title("Stellar rotation period evolution")
axes[1].set_xlim(0, 10)
axes[1].set_ylim(0, 35)
plt.tight_layout()
plt.show()

print(f"At t=0.13 Gyr: P_rot = {(2*np.pi/skumanich_omega(0.13, OMEGA0, TAU_BRAKE))/86400:.2f} d")
print(f"At t=0.60 Gyr: P_rot = {(2*np.pi/skumanich_omega(0.60, OMEGA0, TAU_BRAKE))/86400:.2f} d")
print(f"At t=4.60 Gyr: P_rot = {(2*np.pi/skumanich_omega(4.60, OMEGA0, TAU_BRAKE))/86400:.2f} d")

## Part 2 — Butterfly Diagram via Toy Babcock–Leighton Model / 토이 Babcock–Leighton 나비 도표

**English.** A minimal flux-transport dynamo: dynamo waves migrate equatorward with phase velocity $v_\phi$, so emergence latitude follows $\lambda(t) = \lambda_0 \cos(2\pi t/T_\mathrm{cycle}) \cdot \mathrm{sign}(\cos)$. We sample bipolar emergences with rate proportional to $|\sin(\pi t/T_\mathrm{cycle})|$ and decreasing latitude band over the cycle.

**한국어.** 최소 플럭스 수송 다이나모: 다이나모 파동이 위상 속도 $v_\phi$로 적도방향 이동하므로 출현 위도는 $\lambda(t) = \lambda_0 \cos(2\pi t/T_\mathrm{cycle}) \cdot \mathrm{sign}(\cos)$를 따릅니다. 주기 내에서 출현률이 $|\sin(\pi t/T_\mathrm{cycle})|$에 비례하고 위도 대역이 감소하도록 양극 출현을 표본화합니다.

In [ ]:
def sample_butterfly(n_cycles: int, points_per_cycle: int = 5000) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Sample sunspot emergence times, latitudes, and polarities for a toy butterfly diagram.

    Args:
        n_cycles: Number of 11-year activity cycles to simulate.
        points_per_cycle: Number of candidate emergences per cycle.

    Returns:
        Tuple of (time array in years, latitude array in degrees, polarity array {-1,+1}).
    """
    t_cycle = 11.0
    times, lats, pols = [], [], []
    for k in range(n_cycles):
        phase = rng.uniform(0.0, 1.0, size=points_per_cycle)
        # Activity envelope (probability of emergence)
        prob = np.sin(np.pi * phase) ** 2
        keep = rng.uniform(0, 1, size=points_per_cycle) < prob
        phase = phase[keep]
        # Equatorward drift: starts at +/-30 deg, ends at +/-5 deg
        max_lat = 30.0 - 25.0 * phase
        lat = max_lat * (1.0 - 0.6 * phase) * rng.choice([-1, 1], size=phase.size)
        lat += rng.normal(0.0, 2.0, size=phase.size)  # scatter
        # Hale polarity flips each cycle
        pol = np.where(lat > 0, (-1) ** k, (-1) ** (k + 1))
        times.append(k * t_cycle + phase * t_cycle)
        lats.append(lat)
        pols.append(pol)
    return np.concatenate(times), np.concatenate(lats), np.concatenate(pols)

t_arr, lat_arr, pol_arr = sample_butterfly(n_cycles=4, points_per_cycle=8000)

fig, ax = plt.subplots(figsize=(10, 4.2))
for p, marker, color, label in [(+1, "+", "#d62728", "+ polarity"), (-1, "x", "#1f77b4", "- polarity")]:
    sel = pol_arr == p
    ax.scatter(t_arr[sel], lat_arr[sel], s=4, c=color, marker=marker, alpha=0.5, label=label)
ax.axhline(0, color="k", lw=0.8)
ax.set_xlabel("Time [yr]")
ax.set_ylabel("Latitude [deg]")
ax.set_title("Toy butterfly diagram (4 Hale half-cycles, Schrijver & Zwaan Ch. 6/7)")
ax.legend(loc="upper right", markerscale=2)
ax.set_ylim(-40, 40)
plt.tight_layout()
plt.show()

## Part 3 — Activity–Rotation Diagram with Saturation / 활동–회전 다이어그램과 포화

**English.** Following Ch. 11: $L_X/L_\mathrm{bol}$ saturates at $\sim 10^{-3}$ for $\mathrm{Ro} \le 0.13$, and follows a power law $\propto \mathrm{Ro}^{-2}$ above. We synthesize a population of cool stars and overplot the canonical relation.

**한국어.** 11장에 따라 $L_X/L_\mathrm{bol}$은 $\mathrm{Ro} \le 0.13$에서 $\sim 10^{-3}$로 포화하고, 그 이상에서는 $\propto \mathrm{Ro}^{-2}$의 멱법칙을 따릅니다. 저온 항성 모집단을 합성하여 표준 관계와 함께 그립니다.

In [ ]:
def lx_lbol(ro: np.ndarray, ro_sat: float = 0.13, l_sat: float = 1e-3) -> np.ndarray:
    """Compute X-ray to bolometric luminosity ratio vs. Rossby number.

    Args:
        ro: Rossby number array.
        ro_sat: Saturation threshold.
        l_sat: Saturated value of L_X/L_bol.

    Returns:
        L_X / L_bol array.
    """
    out = np.where(ro <= ro_sat, l_sat, l_sat * (ro / ro_sat) ** -2)
    return out

n_stars = 600
ro_sample = 10 ** rng.uniform(-2.0, 0.7, size=n_stars)
lx_sample = lx_lbol(ro_sample) * 10 ** rng.normal(0.0, 0.2, size=n_stars)  # 0.2 dex scatter

ro_curve = np.logspace(-2.0, 0.7, 400)
fig, ax = plt.subplots(figsize=(8.5, 4.5))
ax.scatter(ro_sample, lx_sample, s=8, alpha=0.4, label="synthetic stars")
ax.loglog(ro_curve, lx_lbol(ro_curve), "k-", lw=2, label="canonical relation")
ax.axvline(0.13, color="r", ls="--", lw=1, label="saturation Ro = 0.13")
ax.axvline(2.0, color="orange", ls=":", lw=1, label="Sun (Ro ~ 2)")
ax.set_xlabel("Rossby number Ro = P_rot / tau_conv")
ax.set_ylabel(r"$L_X / L_\mathrm{bol}$")
ax.set_title("Activity–rotation diagram (Ch. 11)")
ax.legend(loc="lower left")
plt.tight_layout()
plt.show()

## Part 4 — F_X versus Φ Power-Law Relation / F_X 대 Φ 멱법칙

**English.** Reproduce $F_X \propto \Phi^{1.15}$ across nine orders of magnitude (Ch. 8/9). We integrate the cumulative X-ray luminosity over a synthetic flux distribution using `np.trapezoid`.

**한국어.** 9자릿수에 걸친 $F_X \propto \Phi^{1.15}$를 재현합니다(8/9장). `np.trapezoid`를 사용하여 합성 자속 분포에 대해 누적 X선 광도를 적분합니다.

In [ ]:
def fx_phi_relation(phi_mx: np.ndarray, slope: float = 1.15, log_norm: float = -13.5) -> np.ndarray:
    """Pevtsov-style F_X vs. unsigned magnetic flux Phi.

    Args:
        phi_mx: Total unsigned magnetic flux in Maxwell.
        slope: Power-law slope (1.15 from Ch. 8).
        log_norm: log10 of normalization constant.

    Returns:
        X-ray flux in erg cm^-2 s^-1.
    """
    return 10 ** log_norm * phi_mx ** slope

# Categories from monograph Ch. 5
categories = {
    "intranetwork": (1e16, 1e18),
    "network": (1e18, 1e19),
    "ephemeral region": (1e19, 1e20),
    "small AR": (1e20, 1e22),
    "large AR": (1e22, 3e23),
    "stellar (active dwarf)": (1e23, 1e25),
    "RS CVn": (1e25, 1e27),
}

phi_grid = np.logspace(16, 27, 1000)
fx_grid = fx_phi_relation(phi_grid)

# Cumulative emission via np.trapezoid in log space
log_phi = np.log10(phi_grid)
fx_log = np.log10(fx_grid)
cumulative = np.trapezoid(fx_grid, phi_grid)
print(f"Integrated F_X over Phi range = {cumulative:.3e} (arbitrary scaling)")

fig, ax = plt.subplots(figsize=(8.5, 4.6))
ax.loglog(phi_grid, fx_grid, "k-", lw=2, label=r"$F_X = 10^{-13.5} \Phi^{1.15}$")
for name, (lo, hi) in categories.items():
    mask = (phi_grid >= lo) & (phi_grid <= hi)
    ax.plot(phi_grid[mask], fx_grid[mask], lw=4, alpha=0.7, label=name)
ax.set_xlabel(r"Magnetic flux $\Phi$ [Mx]")
ax.set_ylabel(r"$F_X$ [erg cm$^{-2}$ s$^{-1}$]")
ax.set_title("Universal F_X – Phi power law (Ch. 8/9, foreshadowing Pevtsov+2003)")
ax.legend(loc="lower right", fontsize=8)
plt.tight_layout()
plt.show()

## Part 5 — Magnetic Carpet Replacement Time / 자기 카펫 갱신 시간

**English.** Simulate a stochastic ensemble of network elements with exponential lifetimes $\tau_\mathrm{life} = 14$ h. Compute the survival fraction and verify the ~14 h e-folding time predicted in Ch. 5.

**한국어.** 지수적 수명 $\tau_\mathrm{life} = 14$ h을 갖는 네트워크 요소의 확률적 앙상블을 모의실험합니다. 생존 분율을 계산하고 5장에서 예측된 ~14시간 e-접힘 시간을 검증합니다.

In [ ]:
def carpet_simulation(n_elements: int, tau_life_hours: float, t_max_hours: float, n_steps: int) -> tuple[np.ndarray, np.ndarray]:
    """Simulate replacement of magnetic carpet network elements.

    Args:
        n_elements: Number of network elements at t=0.
        tau_life_hours: Mean lifetime in hours.
        t_max_hours: Total simulated duration.
        n_steps: Number of time steps for output.

    Returns:
        Tuple of (time array hours, surviving-original-elements fraction).
    """
    lifetimes = rng.exponential(scale=tau_life_hours, size=n_elements)
    t = np.linspace(0.0, t_max_hours, n_steps)
    survival = np.array([np.mean(lifetimes > ti) for ti in t])
    return t, survival

t_h, surv = carpet_simulation(n_elements=20000, tau_life_hours=14.0, t_max_hours=80.0, n_steps=200)

# Fit e-folding by trapezoidal integration of survival curve
tau_efold_estimate = np.trapezoid(surv, t_h)  # mean lifetime
print(f"Numerical mean lifetime from trapezoid integration: {tau_efold_estimate:.2f} h")
print(f"Expected (input): 14.0 h")

fig, ax = plt.subplots(figsize=(8.5, 4.2))
ax.plot(t_h, surv, lw=2, label="simulated survival")
ax.plot(t_h, np.exp(-t_h / 14.0), "r--", lw=1.5, label=r"$e^{-t/14}$")
ax.axvline(14.0, color="k", ls=":", lw=1, label="tau = 14 h")
ax.set_xlabel("Time [hours]")
ax.set_ylabel("Surviving fraction of original elements")
ax.set_title("Magnetic carpet replacement (Ch. 5)")
ax.legend()
plt.tight_layout()
plt.show()

## Part 6 — Summary / 요약

**English.** This notebook reproduced five quantitative anchors of Schrijver & Zwaan (2000): the Skumanich spin-down law, an equatorward-drifting butterfly diagram with Hale polarity reversal, the saturated activity–rotation diagram, the universal F_X–Φ power law spanning nine flux decades, and the magnetic-carpet replacement timescale. Together these demonstrate the monograph's core claim that solar and stellar magnetic activity is a single, scale-coherent phenomenon governed by dynamo-driven flux generation, surface processing, and rotational evolution.

**한국어.** 이 노트북은 Schrijver & Zwaan(2000)의 다섯 가지 정량적 정초를 재현했습니다. Skumanich 회전 감속 법칙, Hale 극성 반전을 동반한 적도방향 표류 나비 도표, 포화된 활동–회전 다이어그램, 9자릿수 자속 영역에 걸친 보편적 F_X–Φ 멱법칙, 자기 카펫 갱신 시간 척도. 이들은 함께 본 단행본의 핵심 명제 — 태양과 항성의 자기 활동이 다이나모 구동 자속 생성, 표면 처리, 회전 진화에 의해 지배되는 단일하고 척도 일관적인 현상임 — 을 시연합니다.